# 3D Parallelism Simulation Experiments
This notebook replicates the distributed training simulation experiments utilizing 3D Parallelism (Data, Tensor, and Pipeline Parallelism). The simulation mimics real-world distributed cluster execution using standard CPU processes as stand-ins for individual nodes and GPUs.

## Technical Architecture
The core operations are mapped to a virtual 3D coordinate system. The configuration for the current experiment is as follows:
- Data Parallelism (DP) Size: 2
- Tensor Parallelism (TP) Size: 2
- Pipeline Parallelism (PP) Size: 2
- Total World Size: 8 processes


In [ ]:
import os
import torch.multiprocessing as mp
from src.topology.mesh import VirtualDeviceMesh
from src.engine.trainer import SimulatedTrainer
from src.engine.logger import DistributedLogger


## Worker Process Definition
The worker function initializes the virtual device mesh and the simulated trainer. It executes the specified number of training steps, logging the chronological flow of activations and gradients.

To avoid issues with Python multiprocessing in interactive notebook environments, the worker function is imported directly from the `main` module where it is defined.


In [ ]:
# Import the worker function from the main module to ensure it is picklable
# by the multiprocessing framework within a Jupyter notebook environment.
from main import worker


## Execution
We execute the simulation using `torch.multiprocessing.spawn`. We restrict the number of OpenMP threads to 1 to prevent resource contention among the spawned virtual processes.


In [ ]:
def run_simulation() -> None:
    """
    Sets up the configuration for 3D parallelism and spawns the virtual
    processes using torch.multiprocessing.
    """
    dp_size: int = 2
    tp_size: int = 2
    pp_size: int = 2
    world_size: int = dp_size * tp_size * pp_size
    num_steps: int = 2

    print(f"Starting 3D Parallel Simulation with {world_size} processes.")
    print(f"Configuration: DP={dp_size}, TP={tp_size}, PP={pp_size}")

    os.environ["OMP_NUM_THREADS"] = "1"

    mp.spawn(
        worker,
        args=(world_size, dp_size, tp_size, pp_size, num_steps),
        nprocs=world_size,
        join=True
    )

if __name__ == "__main__":
    run_simulation()
